# Reproducing Padilla-Coreano et al. (2022) from dandiset 001749

Paper. Padilla-Coreano N, Batra K, Patarino M, et al. (2022). Cortical ensembles orchestrate
social competition through hypothalamic outputs. *Nature* 603, 667-671.
[doi:10.1038/s41586-022-04507-5](https://doi.org/10.1038/s41586-022-04507-5). The claims file was
written from the PMC author manuscript (PMC9576144) with its Extended Data legends, and the Nature
supplementary information, which holds the Methods.

Dandiset. [DANDI:001749](https://dandiarchive.org/dandiset/001749/0.260311.1727), version
0.260311.1727. The paper cites no version of the dandiset, which was created in 2026, after the
paper. This is the only published version and the most recent one. The draft (modified
2026-09-23) lists the same 105 assets with the same paths, sizes and modification times, so the two
do not differ in content the analysis uses.

What the dandiset holds. 53 NWB files, one per implanted mouse and competition: 10 from the
"ephys" cohort (spikes and tone times only), 21 from the mPFC-LH ("chrimson") cohort and 22 from the
mPFC-BLA cohort (with AlphaTracker poses, port entries and, in 34 files, the nine hand-scored
behaviour labels of Figure 1f). No file holds an alone session, a phototagging light pulse, an
optogenetic stimulation, the unimplanted cohort, or the ex vivo recordings.

| Claim | Tested here |
|---|---|
| ed5b-rank1-wins-most | tested |
| ed5c-rewards-not-different | tested (rewards part; body weights are not in the dandiset) |
| ed5d-subordinates-slower-latency | tested |
| ed5e-port-occupancy-not-different | tested |
| ed5f-dominants-displace-more | tested |
| ed5g-time-displaced-not-different | tested |
| fig1b, fig1c, ed1d, ed1g (unimplanted cohort) | blocked, data missing |
| fig1e-alphatracker-precision | blocked, data missing |
| fig1i, ed5j, ed5k (HMM-GLM) | blocked, not runnable (see the data check) |
| ed7b-distance-by-trial-type-not-rank | blocked, metadata insufficient |
| Single-unit claims of Figures 2 and 3 and ED Figures 6 to 9 (21 claims) | blocked, metadata insufficient |
| ed8ij, fig2e, ed9b (alone sessions) | blocked, data missing |
| fig4a, fig4b (2 claims), ed10c, fig4d, ed10ek (ex vivo, phototagging, optogenetics) | blocked, data missing |

The reasons for each block are in the data check below and in the attempt record.

In [ ]:
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from scipy import stats

import competition as c

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
files = c.load_all()
print(f"loaded {len(files)} files from dandiset {c.DANDISET_ID} version {c.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "file": r["path"].split("/")[0][4:], "cohort": r["cohort"], "mouse": r["mouse"], "competitor": r["competitor"],
    "rank": r["rank"], "other_rank": r["other_rank"], "start": r["session_start"][:16], "units": r["n_units"],
    "rec_s": round(r["es_shape"][0] / r["es_rate"]), "last_spike_s": round(r["last_spike"]),
    "cues": len(r["cs"]), "last_cue_s": round(r["cs"][-1]), "deliveries": len(r["us"]),
    "port_entries": None if r["pe_start"] is None else len(r["pe_start"]),
    "label_trials": None if r["labels"] is None else r["labels"].shape[0],
    "pair_in_dataset": r["pair_in_dataset"]} for r in files])
print(info.to_string(index=False))
print("\nFiles per cohort:", info.cohort.value_counts().to_dict())
print("Mice:", info.mouse.nunique(), "; per cohort:", info.groupby("cohort").mouse.nunique().to_dict())
sessions = {c.session_key(r) for r in files}
print("Competition sessions (date in file name, cage, pair):", len(sessions))
both = [k for k in sessions if sum(c.session_key(r) == k for r in files) == 2]
print("Sessions with the files of both mice:", len(both))
print("Files whose pair_in_dataset is 'not included':", (info.pair_in_dataset == "not included").sum(), "of", len(info))
print("Recording channels and rate:", sorted({(r["es_shape"][1], r["es_rate"], r["es_unit"]) for r in files}))

The 53 files hold 20 mice: 5 in the "ephys" cohort, 7 mPFC-LH and 8 mPFC-BLA. The paper's mPFC
recordings come from 20 mice with cells. Every file says `pair_in_dataset: not included`, but for
the sessions counted above both mice of the pair are in the dandiset; the field is wrong for those
files. The session count uses the date in the file name. Some file names and session start times
disagree; see the next cell.

In [ ]:
# Ranks: one rank per mouse across files, and the competitor's rank as its own file gives it.
ranks = {}
conflicts = []
for r in files:
    for m, k in ((r["mouse"], r["rank"]), (r["competitor"], r["other_rank"])):
        if ranks.setdefault(m, k) != k:
            conflicts.append((m, ranks[m], k))
print("Mice with more than one rank across files:", conflicts)
print("Rank per mouse:", dict(sorted(ranks.items())))
mismatch = [(r["path"].split("/")[0], r["description"][:95]) for r in files
            if (r["desc_animal"], r["desc_other"]) != (r["animal"], r["other_animal"])]
print("\nFiles whose subject description names other animals than the file name:")
for m in mismatch:
    print("  ", m)
dates = [(r["path"].split("/")[0], r["session_start"]) for r in files if r["session_start"][:10].replace("-", "") != r["date"]]
print("\nFiles whose session start date differs from the date in the file name:", dates)
print("\nCompetitions per own rank (53 files):", info["rank"].value_counts().sort_index().to_dict())

Every mouse has one rank across files, and the competitor's rank in a file agrees with the
competitor's own files. In one file the subject description names mouse 2 against mouse 3 while the
file name says mouse 3 against mouse 2; the ranks it gives (4 for the subject, 1 for the competitor)
are those of mice 3 and 2 elsewhere, so the description swaps the animal numbers but keeps the
ranks of the file's subject. One file named 20200121 has a session start of 2020-01-22 13:30. Its
labels mirror those of the file of its competitor in the 2020-01-21 13:15 session (shown below), so
the start time is wrong.

In [ ]:
schedules = {tuple(np.round(r["cs"], 2)) for r in files}
shared = 0
while all(len(x) > shared for x in schedules) and len({x[shared] for x in schedules}) == 1:
    shared += 1
print("Distinct tone schedules:", len(schedules), "; leading tone onsets shared by all 53 files:", shared)
for cohort_is_ephys in (True, False):
    sub = {tuple(np.round(r["cs"], 2)) for r in files if (r["cohort"] == "mPFC") == cohort_is_ephys}
    longest = max(sub, key=len)
    print(f"  {'ephys' if cohort_is_ephys else 'mPFC-LH and mPFC-BLA'} files: {len(sub)} schedules, all prefixes of the longest ({len(longest)} tones):",
          all(x == longest[:len(x)] for x in sub), "; ITIs (s)", np.diff(longest).min(), "to", np.diff(longest).max())
print("Tone durations (s):", sorted({round(float(d), 3) for r in files for d in r["cs_stop"] - r["cs"]}))
d = np.concatenate([r["us"] - r["cs"][np.searchsorted(r["cs"], r["us"]) - 1] for r in files])
print("Delivery minus the tone onset before it (s):", sorted(set(np.round(d, 3))), "; the Methods give 2 s")
print("Files with fewer deliveries than tones:", [(r["path"][4:28], len(r["cs"]), len(r["us"])) for r in files if len(r["us"]) != len(r["cs"])])
print("Files whose last tone is after the end of the recording:",
      int((info.last_cue_s > info.rec_s).sum()), "of", len(info), info.loc[info.last_cue_s > info.rec_s, "cohort"].value_counts().to_dict())
lab = info.dropna(subset=["label_trials"])
print("Labelled files with fewer label trials than tones:", int((lab.label_trials < lab.cues).sum()), "of", len(lab))

All 53 files share their first 17 tone onsets (60.01, 140.01, 230.01 s and so on); after them the
"ephys" files follow one schedule and the other files another, each cut at different lengths, so the
tone times are the program's schedule, not times recorded in each session. The port entries and labels agree with it
(below), so for the files with port entries it is the schedule that was run, in the behaviour clock.
The deliveries are 0.49 s after tone onset in every file, where the Methods give 2 s; the analysis
below takes 2 s from the Methods and shows 0.49 s as a variant. In 7 of the 10 "ephys" files the
last tones fall after the end of the neural recording. Five files have fewer deliveries than tones. The labels hold fewer trials than tones in some files;
label trial *j* is taken to be tone *j*, which the agreement with the port entries below supports.

In [ ]:
pe_files = [r for r in files if r["labels"] is not None and r["pe_start"] is not None]
rows = []
for r in pe_files:
    selfin = np.isin(r["labels"], c.SELF_IN).astype(float)
    n = min(len(r["cs"]), selfin.shape[0])
    def corr(lag):
        occ = c.port_occupancy_on_label_grid(r, lag)
        return np.corrcoef(selfin[:n].ravel(), occ[:n].ravel())[0, 1]
    occ = c.port_occupancy_on_label_grid(r, 0.0)
    shifted = np.corrcoef(selfin[1:n].ravel(), occ[:n - 1].ravel())[0, 1]   # labels of trial j+1 against entries of trial j
    scan = {lag: corr(lag) for lag in np.arange(-3, 3.01, 0.25)}
    best = max(scan, key=scan.get)
    rows.append({"file": r["path"][4:28], "r_lag0": corr(0.0), "best_lag_s": best, "r_best": scan[best], "r_next_trial": shifted})
pl = pd.DataFrame(rows)
print(pl.round(3).to_string(index=False))
print("\nMedian r at lag 0:", round(pl.r_lag0.median(), 3), "; median r with labels of the next trial:", round(pl.r_next_trial.median(), 3),
      "; files whose best lag within +-3 s is within 0.5 s of 0:", int((pl.best_lag_s.abs() <= 0.5).sum()), "of", len(pl))

The subject's port entries (a lickometer or beam signal) and the hand-scored "self in port" labels
agree at lag 0 (correlation over the label bins of each trial; the contrast is the same correlation
with the labels of the next trial). This shows that the tone schedule, the port entries and the
labels share one clock, the behaviour clock, in the files with both, and that label trial *j* is
tone *j*. The files whose correlation is low are named in the table.

In [ ]:
al = c.assets()
started = time.time()
with ThreadPoolExecutor(8) as ex:
    clocks = pd.DataFrame(list(ex.map(c.clock_checks, al)))
clocks["file"] = clocks.path.str.split("/").str[0].str[4:]
print(f"clock checks of {len(clocks)} files in {time.time() - started:.0f} s")
cols = ["file", "raw_peak_sd_1.0", "raw_lag_1.0", "raw_peak_sd_1.5", "raw_lag_1.5"]
print(clocks[cols].round(3).to_string(index=False))
print("\nPeak of the spike/raw cross-correlation (SD over lags), median over files: stored times",
      round(clocks["raw_peak_sd_1.0"].median(), 1), "; times x 1.5", round(clocks["raw_peak_sd_1.5"].median(), 1))
print("Files with a peak above 10 SD: stored", int((clocks["raw_peak_sd_1.0"] > 10).sum()), "; x 1.5", int((clocks["raw_peak_sd_1.5"] > 10).sum()))
print("Last spike / recording length, range over files:", round((info.last_spike_s / info.rec_s).min(), 3), "to", round((info.last_spike_s / info.rec_s).max(), 3))
off = clocks[(clocks["raw_peak_sd_1.5"] > 10) & (clocks["raw_lag_1.5"].abs() > 0.1)]
print("Files that line up only after an offset of more than 0.1 s (x 1.5):", len(off), "; offsets (s) from", round(off["raw_lag_1.5"].min(), 2), "to", round(off["raw_lag_1.5"].max(), 2))
print("Files without a peak above 10 SD at either scale:", clocks.loc[(clocks["raw_peak_sd_1.5"] <= 10) & (clocks["raw_peak_sd_1.0"] <= 10), "file"].tolist())

The stored spike times do not line up with the raw recording: their cross-correlation with threshold
crossings of 20 s of the band-passed `ElectricalSeries` (offsets of -60 to +60 s) has no peak above
the 3 to 7 SD of chance. The same times multiplied by 1.5 line up, at a lag of a few milliseconds in
most files, and in every file the last spike falls at about two thirds of the recording length. The
spike times were divided by 30 kHz where the logger samples at 20 kHz (as the file states). This is
a format error; times x 1.5 are used below. In the files listed above as lining up only after an
offset, the scaled spikes match the recording shifted by seconds, so in those files the sorted
spikes and the raw recording do not share a start either.

In [ ]:
pe = clocks.dropna(subset=["pe_lag"]).copy()
pe["clear"] = pe.pe_peak > 2 * pe.pe_p99
cols = ["file", "n_entries", "pe_lag", "pe_peak", "pe_p99", "pe_median", "pe_at_zero", "cs_at_pe_lag", "cs_p99", "clear"]
print(pe[cols].round(2).to_string(index=False))
print("\nFiles with port entries:", len(pe), "; with a clear peak (more than twice the 99th percentile over offsets):", int(pe.clear.sum()))
print("Offsets of the clear peaks (s): median", round(pe[pe.clear].pe_lag.median(), 1), "range", round(pe[pe.clear].pe_lag.min(), 1), "to", round(pe[pe.clear].pe_lag.max(), 1))
print("Score at offset 0 against the 99th percentile:", int((pe.pe_at_zero > pe.pe_p99).sum()), "files above")
by_day = pe[pe.clear].assign(day=pe.file.str[:8] + "-" + pe.file.str.extract(r"-(\w+\d)-")[0]).groupby("day").pe_lag.agg(["count", "min", "max"])
print("\nClear offsets by day and cage:\n", by_day.round(1).to_string())

In [ ]:
# Recordings deposited twice: the two files of a session share the video and poses; compare their
# recordings, sample counts and units, and look for equal sample counts across all files.
url = {a["path"]: a["url"] for a in al}
rows = []
for k in sorted(lab_key for lab_key in {c.session_key(r) for r in files}):
    v = [r for r in files if c.session_key(r) == k]
    if len(v) != 2:
        continue
    fa, fb = c.open_nwb(url[v[0]["path"]]), c.open_nwb(url[v[1]["path"]])
    xa = fa["acquisition/ElectricalSeries/data"][6_000_000:6_002_000, :]
    xb = fb["acquisition/ElectricalSeries/data"][6_000_000:6_002_000, :]
    rows.append({"session": "-".join(k), "samples_a": v[0]["es_shape"][0], "samples_b": v[1]["es_shape"][0],
                 "same_block": bool(np.array_equal(xa, xb)), "units_a": v[0]["n_units"], "units_b": v[1]["n_units"],
                 "same_spike_counts": bool(np.array_equal(np.sort(v[0]["spike_counts"]), np.sort(v[1]["spike_counts"])))})
dup = pd.DataFrame(rows)
print(dup.to_string(index=False))
print("\nPairs of files of one session with identical recordings or spike counts:", int((dup.same_block | dup.same_spike_counts).sum()))

import hashlib
def fingerprint(a):
    d = c.open_nwb(a["url"])["acquisition/ElectricalSeries/data"]
    return a["path"].split("/")[0][4:], d.shape[0], hashlib.md5(d[2_000_000:2_000_500, :].tobytes()).hexdigest()
with ThreadPoolExecutor(12) as ex:
    fps = pd.DataFrame(list(ex.map(fingerprint, al)), columns=["file", "samples", "md5_of_500_samples"])
groups = fps[fps.duplicated("md5_of_500_samples", keep=False)]
print("Files whose recordings share a block of 500 samples at 100 s (all 53 files):")
print(groups.to_string(index=False))
same = [r for r in files if r["path"].split("/")[0][4:] in set(groups.file)]
if len(same) == 2:
    f0, f1 = c.open_nwb(url[same[0]["path"]]), c.open_nwb(url[same[1]["path"]])
    print("Identical at samples 1e6, 3e7 and 6e7:", [bool(np.array_equal(f0["acquisition/ElectricalSeries/data"][o:o + 2000, :], f1["acquisition/ElectricalSeries/data"][o:o + 2000, :])) for o in (1_000_000, 30_000_000, 60_000_000)],
          "; units", [r["n_units"] for r in same], "; session starts", [r["session_start"][:16] for r in same])

Within each session the two loggers hold different signals, lengths and clusters; the two files of
a session share the behaviour video and its poses, and their labels are two views of one scoring
(below). Across all files, one recording is deposited twice: the `ElectricalSeries` of
sub-20191025-chrimson2-3-1v3 and sub-20191029-chrimson2-3-1v3 (the same mouse against the same
competitor, four days apart) are the same signal. Their units differ, and the 20191025 file's units
match its recording at no offset (the spike check above), so the 20191025 file most likely carries
the 20191029 recording in place of its own. Neither file is labelled, so no behavioural test below
uses them.

The spikes (x 1.5) and the behaviour clock are offset. Placing the port entries on the spike clock
at each offset from -300 to +300 s, the units respond to them at a single offset, different in each
session (from a few seconds to tens of seconds), and not at offset 0. Files recorded on the same day
in the same cage have nearly the same offset; the tones give a weaker response at the same offset.
The dandiset description says this offset is in a `video_gap` variable of the behavioural events;
no file holds such a variable. For the "ephys" files, which have no port entries, it cannot be
estimated this way, and for the files without a clear peak it cannot be estimated at all.

In [ ]:
q = np.concatenate([r["spike_counts"] / (r["es_shape"][0] / r["es_rate"]) for r in files])
print("Clusters:", len(q), "; quality labels:", sorted({x for r in files for x in r["quality"]}))
print("Firing rate (Hz) percentiles 5, 25, 50, 75, 95:", np.percentile(q, [5, 25, 50, 75, 95]).round(2))
print("Clusters at or above 0.05 Hz:", int((q >= 0.05).sum()), "; above 1 Hz:", int((q > 1).sum()), "; above 100 Hz:", int((q > 100).sum()))
dom = sum(r["n_units"] for r in files if r["rank"] < r["other_rank"]); sub = sum(r["n_units"] for r in files if r["rank"] > r["other_rank"])
print("Clusters in files of the relative dominant:", dom, "; of the relative subordinate:", sub)
print("Paper: 998 cells (507 dom, 490 sub in Figure 2a), 913 kept for clustering")

The `units` tables hold every Klusta cluster, 5,879 in all, each labelled `unsorted`, where the
paper analyses 998 manually curated cells. The table has the column for a curation label (good, mua,
noise) but no cluster carries one, and no simple rule on firing rate comes near 998. The paper's
cells cannot be told apart from multi-unit and noise clusters, so every claim on single units is
blocked as `metadata_insufficient`.

In [ ]:
lf = [r for r in files if r["labels"] is not None]
print("Labelled files:", len(lf), "; mice:", len({r["mouse"] for r in lf}), "; sessions:", len({c.session_key(r) for r in lf}),
      "; trials:", sum(r["labels"].shape[0] for r in lf))
print("Label bins per trial:", sorted({r["labels"].shape[1] for r in lf}), "at", sorted({r["label_rate"] for r in lf}), "Hz")
print("Label trials in files of the relative dominant:", sum(r["labels"].shape[0] for r in lf if r["rank"] < r["other_rank"]),
      "; of the relative subordinate:", sum(r["labels"].shape[0] for r in lf if r["rank"] > r["other_rank"]))
labelled_names = {r["path"].split("/")[0][4:] for r in lf}
print("Labelled files with a clear offset between spikes and port entries (data check above):",
      len(labelled_names & set(pe[pe.clear].file)), "of", len(labelled_names))
allv = np.concatenate([r["labels"].ravel() for r in lf])
print("Label codes and bins:", dict(zip(*np.unique(allv, return_counts=True))))
# Pairs: how the two views of one session relate.
by = {}
for r in lf:
    by.setdefault(c.session_key(r), []).append(r)
rows, co = [], {}
for k, v in sorted(by.items()):
    if len(v) == 2 and v[0]["labels"].shape == v[1]["labels"].shape:
        A, B = v[0]["labels"], v[1]["labels"]
        pos = np.isin(A, list(c.MIRROR_POSITION))
        agree = np.mean(np.vectorize(lambda x: c.MIRROR_POSITION.get(x, -1))(A[pos]) == B[pos])
        rows.append({"session": "-".join(k), "position bins agreeing": round(float(agree), 3)})
        for a, b in zip(A.ravel(), B.ravel()):
            if a in (1, 2, 3) or b in (1, 2, 3):
                co[(int(a), int(b))] = co.get((int(a), int(b)), 0) + 1
print(pd.DataFrame(rows).to_string(index=False))
print("\nPush, resist and displaced bins in one view (first) against the other view (second):", dict(sorted(co.items())))

The 34 labelled files hold 965 trials from 14 mice, 280 bins at 20 Hz each (-2 to +12 s from tone
onset, as the description says), which matches the 965 trials of the paper's HMM-GLM dataset
(Methods 17; 13 mice, 32 sessions). Where both mice of a session are labelled, one file's labels are
the other's seen from the competitor (self in port against other in port, and so on); most pairs
agree in nearly every bin, two in only 64% and 75%. A push (1) in one view is displaced (3) or
resist (2) in the other, and resist and displaced are pushes in the other view, so one file shows the
outcome of the competitor's pushes but not of its subject's own. Labels change in whole seconds (runs of 20 bins).

In [ ]:
with ThreadPoolExecutor(8) as ex:
    pose = pd.DataFrame(list(ex.map(c.pose_summary, [a for a in al if "ephys" not in a["path"].split("/")[0]])))
pose["file"] = pose.path.str.split("/").str[0].str[4:]
print(pose.drop(columns=["path"]).round(3).to_string(index=False))
print("\nFiles whose poses are all missing (both mice):", int(((pose.Mouse1_nan == 1) & (pose.Mouse2_nan == 1)).sum()), "of", len(pose))
print("Body parts:", pose.parts.unique())

AlphaTracker poses are given for two mice, `Mouse1` and `Mouse2`, with ears, tail base and logger
(no nose, which the Methods say was tracked). Nothing says which of them is the file's subject, and
the files of many sessions hold no position at all (all values missing). The port location the paper
marked by hand is not in the files. Distance to the port (ED Fig. 7b) cannot be computed.

## Trial outcomes

The claims below use the trial outcomes of the labelled files, from the Methods ("Reward
competition behavioral analysis", quoted in `competition.py`). Relative dominant (dom) is the
mouse of the pair with the higher tube-test rank (the smaller number).

In [ ]:
def outcomes(r, delivery_s=c.DELIVERY_S, switch_rule="first_last"):
    return c.trial_outcomes(r["labels"], delivery_s, switch_rule)

rows = []
for r in lf:
    for name, kw in {"main": {}, "delivery 0.49 s": {"delivery_s": 0.49}, "switch = both in port": {"switch_rule": "any"}}.items():
        o = outcomes(r, **kw)
        rows.append({"reading": name, "file": r["path"][4:28], "win": int((o == "win").sum()), "lose": int((o == "lose").sum()), "excluded": int((o == "excluded").sum())})
oc = pd.DataFrame(rows)
print(oc.groupby("reading")[["win", "lose", "excluded"]].sum().to_string())

### ed5b-rank1-wins-most

> In the recording cohort, rank 1 mice won the highest percentage of competitions. (ED Fig. 5b;
> expected 12, 12, 15 and 14 competitions for ranks 1 to 4.)

A competition is won by the mouse that won more of its trials. A file's competition is decided from
its own labels or, when it has none, from the labels of its competitor's file (whose losses are its
wins).

In [ ]:
lab_by_session = {}
for r in lf:
    lab_by_session.setdefault(c.session_key(r), []).append(r)

def competition_result(r, **kw):
    if r["labels"] is not None:
        o = outcomes(r, **kw); w, l = (o == "win").sum(), (o == "lose").sum()
        src = "own labels"
    else:
        others = [x for x in lab_by_session.get(c.session_key(r), []) if x["mouse"] == r["competitor"]]
        if not others:
            return None, "no labels"
        o = outcomes(others[0], **kw); w, l = (o == "lose").sum(), (o == "win").sum()
        src = "competitor's labels"
    return ("won" if w > l else "lost" if l > w else "tie"), src

for name, kw in {"main": {}, "delivery 0.49 s": {"delivery_s": 0.49}, "switch = both in port": {"switch_rule": "any"}}.items():
    res = pd.DataFrame([{"file": r["path"][4:28], "rank": r["rank"], "result": competition_result(r, **kw)[0], "source": competition_result(r, **kw)[1]} for r in files])
    known = res.dropna(subset=["result"])
    t = known.groupby("rank").result.value_counts().unstack(fill_value=0)
    for col in ("won", "lost", "tie"):
        if col not in t:
            t[col] = 0
    t["competitions (all files)"] = res.groupby("rank").size()
    t["% won, tie as not won"] = (100 * t.won / (t.won + t.lost + t.tie)).round(1)
    t["% won, ties left out"] = (100 * t.won / (t.won + t.lost)).round(1)
    print(f"--- {name}")
    if name == "main":
        print("Sources:", res.source.value_counts().to_dict())
    print(t.to_string())
    for col in ("% won, tie as not won", "% won, ties left out"):
        print(f"  highest {col}: rank {int(t[col].idxmax())}")

**Result: reproduced.** Rank 1 mice won 10 of the 11 competitions whose outcome the labels give
(90.9%), against 12.5% for rank 2, 44.4% for rank 3 and 62.5% for rank 4, under both readings of a tie
(there are none) and under the variants of the trial rule. The competitions per rank over the 53
files (12, 11, 15, 15) are within tolerance of the expected 12, 12, 15 and 14. Only 36 of the 53
competitions have labels, of the file's subject or its competitor.

### ed5c-rewards-not-different

> Among mice wearing loggers, relative dominants and subordinates obtained similar numbers of
> rewards, while the body weight difference between competitors correlated with rewards won.

Rewards are trials won. Readings of the pairs: (a) each labelled file, with its subject and its
competitor (the paper's 22 pairs and its other "session" counts are closer to files than to
sessions; see the data check); (b) each session once, from the dominant's file when there is one.
Body weights are not in the dandiset, so the correlation is not tested.

In [ ]:
def per_file(fn):
    rows = []
    for r in lf:
        s, o = fn(r)
        dom_self = r["rank"] < r["other_rank"]
        rows.append({"file": r["path"][4:28], "session": c.session_key(r), "dom_file": dom_self,
                     "dom": s if dom_self else o, "sub": o if dom_self else s,
                     "dom_mouse": r["mouse"] if dom_self else r["competitor"], "sub_mouse": r["competitor"] if dom_self else r["mouse"]})
    return pd.DataFrame(rows)

def per_session(d):
    return d.sort_values("dom_file", ascending=False).groupby("session").first().reset_index()

def paired(d, label):
    d = d.dropna(subset=["dom", "sub"])
    t = stats.ttest_rel(d["dom"], d["sub"])
    diff = d["dom"] - d["sub"]
    print(f"{label}: n = {len(d)}, dom mean {d['dom'].mean():.2f}, sub mean {d['sub'].mean():.2f}, "
          f"mean difference {diff.mean():.2f} (95% CI {diff.mean() - 1.96 * diff.std(ddof=1) / np.sqrt(len(d)):.2f} to "
          f"{diff.mean() + 1.96 * diff.std(ddof=1) / np.sqrt(len(d)):.2f}), paired t = {t.statistic:.2f}, p = {t.pvalue:.3g}")
    return t.pvalue

def rewards(r, **kw):
    o = outcomes(r, **kw)
    return (o == "win").sum(), (o == "lose").sum()

rw = per_file(rewards)
p_a = paired(rw, "(a) files")
p_b = paired(per_session(rw), "(b) sessions")
paired(per_file(lambda r: rewards(r, delivery_s=0.49)), "(a) files, delivery at 0.49 s")
paired(per_file(lambda r: rewards(r, switch_rule="any")), "(a) files, switch = both in port")
print("\nRewards by dominant mouse (reading a):")
print(rw.groupby("dom_mouse")[["dom", "sub"]].agg(["count", "sum"]).to_string())

**Result: reproduced (the rewards part).** Dominants and subordinates did not differ in rewards by the
paired t-test: dominants 14.12 and subordinates 12.38 rewards over 34 files (p = 0.505), and 14.35
against 12.15 over 20 sessions (p = 0.498). The correlation with body weight could not be tested, as
the dandiset holds no body weights.

### ed5d-subordinates-slower-latency

> In win trials, relative subordinates had longer latencies to pick up the reward than relative
> dominants. (Wilcoxon rank-sum p = 0.012; 326 dom and 358 sub trials.)

Each file's subject in its own win trials, from its port entries (the criterion's "latency of the
winner's port entry"): seconds from tone onset to the first port entry, 0 when the mouse is in the
port at onset (reading a; reading b, from delivery at 2 s, differs by a constant and gives the same
tests). How a mouse already in the port is counted is not given; the variant takes the first entry
that starts after onset. A second variant takes the latency from the hand-scored labels (first bin
labelled self in port at or after onset).

In [ ]:
def entry_latency(r, j, in_port_is_zero=True):
    t = r["cs"][j]
    s, e = r["pe_start"], r["pe_stop"]
    if in_port_is_zero and np.any((s <= t) & (e > t)):
        return 0.0
    k = s[s >= t]
    return float(k[0] - t) if len(k) and k[0] - t <= 12 else np.nan

rows = []
for r in lf:
    o = outcomes(r)
    lat = c.latency_to_port(r["labels"], "self")
    for j in np.flatnonzero(o == "win"):
        rows.append({"mouse": r["mouse"], "file": r["path"][4:28], "group": "dom" if r["rank"] < r["other_rank"] else "sub",
                     "latency_labels": lat[j],
                     "latency_entries": entry_latency(r, j) if r["pe_start"] is not None and j < len(r["cs"]) else np.nan,
                     "latency_next_entry": entry_latency(r, j, False) if r["pe_start"] is not None and j < len(r["cs"]) else np.nan})
lt = pd.DataFrame(rows)
for col in ("latency_entries", "latency_next_entry", "latency_labels"):
    d = lt.dropna(subset=[col])
    a, b = d[d.group == "dom"][col], d[d.group == "sub"][col]
    rs = stats.mannwhitneyu(b, a, alternative="two-sided")
    ks = stats.ks_2samp(a, b)
    print(f"{col}: dom n = {len(a)}, median {a.median():.2f} s, mean {a.mean():.2f}; sub n = {len(b)}, median {b.median():.2f} s, mean {b.mean():.2f}; "
          f"rank-sum p = {rs.pvalue:.3g} (U for sub > dom = {rs.statistic:.0f}, common-language effect {rs.statistic / (len(a) * len(b)):.2f}); KS p = {ks.pvalue:.3g}")
print("\nRange (s): entries", lt.latency_entries.min(), "to", round(lt.latency_entries.max(), 2), "; labels", lt.latency_labels.min(), "to", lt.latency_labels.max())
print("Win trials without a port entry within 12 s, or in a file without port entries:", int(lt.latency_entries.isna().sum()), "of", len(lt))
print("\nPer mouse and group:")
print(lt.groupby(["group", "mouse"])[["latency_entries", "latency_labels"]].agg(["count", "median"]).round(2).to_string())
for col in ("latency_entries", "latency_labels"):
    mm = lt.groupby(["group", "mouse"])[col].mean().reset_index().dropna()
    print(f"Mouse means of {col}, sub against dom (rank-sum over {len(mm)} mouse groups): p =",
          round(stats.mannwhitneyu(mm[mm.group == "sub"][col], mm[mm.group == "dom"][col]).pvalue, 3))

**Result: reproduced.** In their win trials, subordinates entered the port later than dominants
(port entries; median 1.47 against 1.01 s over 189 and 211 trials, rank-sum p = 0.0196, KS p =
0.0036), and also when a mouse already in the port is given its next entry (p = 0.0204). From the
hand-scored labels the difference is in the same direction but the rank-sum is not significant
(median 1.55 against 1.17 s, p = 0.206; KS p = 0.027). Over mouse means the rank-sum gives p = 0.055.

### ed5e-port-occupancy-not-different

> Percent port occupancy during the tone did not differ by relative rank in the recording cohort.
> (paired t-test p = 0.90, 31 sessions)

Percent of the 10 s tone (label bins 0 to 10 s) labelled "self in port" (4, 5) and "other in port"
(6, 7), dom against sub, under readings (a) files and (b) sessions.

In [ ]:
tone = slice(int(-c.LABEL_START_S * c.LABEL_HZ), int((c.TONE_S - c.LABEL_START_S) * c.LABEL_HZ))
occ = per_file(lambda r: (100 * np.isin(r["labels"][:, tone], c.SELF_IN).mean(), 100 * np.isin(r["labels"][:, tone], c.OTHER_IN).mean()))
paired(occ, "(a) files")
paired(per_session(occ), "(b) sessions")
print("Range of percent occupancy:", round(min(occ["dom"].min(), occ["sub"].min()), 1), "to", round(max(occ["dom"].max(), occ["sub"].max()), 1))

**Result: reproduced.** Port occupancy during the tone did not differ by relative rank: dominants
25.19% and subordinates 20.72% over 34 files (paired t-test p = 0.327), 26.37% against 20.23% over 20
sessions (p = 0.284).

### ed5f-dominants-displace-more

> Relative dominants were more successful at displacing subordinates from the port, over the whole
> competition and during the tone. (paired t-tests, p = 0.002, 32 sessions; p = 0.005, 31 sessions)

Pushing success is the percent of a mouse's pushes that displaced the competitor (Methods). A push
is a run of bins labelled 1 in the pusher's file; it succeeded when the competitor's file labels any
of its bins 3. This needs the files of both mice, so the test runs over the sessions that have both.
A variant reads each mouse's pushes from its competitor's file (runs of 2 or 3; success when a 3).
"Whole competition" is all labelled bins (-2 to +12 s of each trial), as the labels cover only the
trials; "tone" is 0 to 10 s.

In [ ]:
pairs = [v for v in lab_by_session.values() if len(v) == 2]
def success(n, s):
    return 100 * s / n if n else np.nan
rows = []
for v in pairs:
    d0, s0 = (v[0], v[1]) if v[0]["rank"] < v[0]["other_rank"] else (v[1], v[0])
    n_t = min(d0["labels"].shape[0], s0["labels"].shape[0])
    D, S = d0["labels"][:n_t], s0["labels"][:n_t]
    row = {"session": "-".join(c.session_key(d0)), "dom_mouse": d0["mouse"], "sub_mouse": s0["mouse"]}
    for name, b in (("whole", slice(None)), ("tone", tone)):
        nd, sd = c.push_success(D, S, b); ns, ss = c.push_success(S, D, b)
        row[f"{name}_dom_pushes"], row[f"{name}_dom"] = nd, success(nd, sd)
        row[f"{name}_sub_pushes"], row[f"{name}_sub"] = ns, success(ns, ss)
        nd2, sd2 = c.competitor_push_success(S, b); ns2, ss2 = c.competitor_push_success(D, b)
        row[f"{name}_dom_v"], row[f"{name}_sub_v"] = success(nd2, sd2), success(ns2, ss2)
    rows.append(row)
ps = pd.DataFrame(rows)
print(ps.round(1).to_string(index=False))
for name in ("whole", "tone"):
    for suffix, lab_ in (("", "pusher's view"), ("_v", "competitor's view")):
        d = ps.rename(columns={f"{name}_dom{suffix}": "dom", f"{name}_sub{suffix}": "sub"})
        paired(d, f"{name}, {lab_}")

**Result: reproduced.** Over the 13 sessions with both files and pushes by both mice, dominants
displaced the competitor in 91.7% of their pushes and subordinates in 60.2% over the labelled bins
(paired t-test p = 0.0051), and 93.4% against 56.5% during the tone (p = 0.0013). Both views of a
push give the same numbers.

### ed5g-time-displaced-not-different

> Total time being displaced from the port did not differ by relative rank in the recording cohort.
> (paired t-test p = 0.15, 31 sessions)

Seconds labelled 3 (displaced) in each mouse's own file, over the labelled bins, dom against sub,
over the sessions with both files, and during the tone only as a variant. A second variant compares
the files of dominants and subordinates unpaired, over all labelled files.

In [ ]:
rows = []
for v in pairs:
    d0, s0 = (v[0], v[1]) if v[0]["rank"] < v[0]["other_rank"] else (v[1], v[0])
    rows.append({"session": "-".join(c.session_key(d0)), "dom": (d0["labels"] == 3).sum() / c.LABEL_HZ, "sub": (s0["labels"] == 3).sum() / c.LABEL_HZ,
                 "dom_tone": (d0["labels"][:, tone] == 3).sum() / c.LABEL_HZ, "sub_tone": (s0["labels"][:, tone] == 3).sum() / c.LABEL_HZ})
tdp = pd.DataFrame(rows)
print(tdp.round(2).to_string(index=False))
paired(tdp, "time displaced (s), own file")
paired(tdp.drop(columns=["dom", "sub"]).rename(columns={"dom_tone": "dom", "sub_tone": "sub"}), "time displaced during the tone (s)")
one = per_file(lambda r: ((r["labels"] == 3).sum() / c.LABEL_HZ, np.nan))
d = one.assign(t=np.where(one.dom_file, one["dom"], one["sub"]))
a, b = d[d.dom_file].t, d[~d.dom_file].t
print(f"Unpaired variant over all {len(d)} labelled files: dom n = {len(a)} mean {a.mean():.2f} s, sub n = {len(b)} mean {b.mean():.2f} s, Welch p = {stats.ttest_ind(a, b, equal_var=False).pvalue:.3g}")

**Result: reproduced.** Time displaced did not differ by relative rank over the 14 sessions with both
files: dominants 3.21 s and subordinates 3.93 s over the labelled bins (paired t-test p = 0.359), and
2.71 against 3.14 s during the tone (p = 0.512).

## Summary

| Claim | Outcome |
|---|---|
| ed5b-rank1-wins-most | reproduced |
| ed5c-rewards-not-different | reproduced (rewards part; no body weights in the dandiset) |
| ed5d-subordinates-slower-latency | reproduced (from port entries; not significant from the labels) |
| ed5e-port-occupancy-not-different | reproduced |
| ed5f-dominants-displace-more | reproduced |
| ed5g-time-displaced-not-different | reproduced |
| fig1b, fig1c, ed1d, ed1g, fig1e | blocked, data missing (unimplanted cohort, AlphaTracker evaluation frames) |
| fig1i, ed5j, ed5k | blocked, not runnable (HMM-GLM) |
| ed7b-distance-by-trial-type-not-rank | blocked, metadata insufficient (no port location, no pose identity) |
| 21 single-unit claims of Figures 2 and 3 and ED Figures 6 to 9 | blocked, metadata insufficient (curated cells not marked) |
| ed8ij, fig2e, ed9b | blocked, data missing (alone sessions) |
| fig4a, fig4b-phototagged-counts, fig4b-lh-stronger-reward-excitation, ed10c, fig4d, ed10ek | blocked, data missing |

How the analysis differs from the paper's. The behavioural tests use the hand-scored labels of
the 34 labelled files (and the port entries for latency), not the paper's AlphaTracker distance or its
GUI scoring of whole sessions; the labels cover -2 to +12 s of each trial, so "whole competition"
means the labelled bins. The paper's "sessions" (31 or 32) are closer to the 34 files than to the 20
sessions; both are shown where the measure allows. Pushing success and time displaced need both
mice's files and use 13 and 14 sessions. Reward delivery is taken at 2 s, from the Methods; the files
give 0.49 s.

What was found about the dataset. The stored spike times are two thirds of their time in the
recording (divided by 30 kHz, with a 20 kHz recording). In 18 of the 21 mPFC-LH files and one
"ephys" file the scaled spikes match the recording only after an offset of 5 to 28 s, and in the
other 3 mPFC-LH files at no offset within 60 s. The offset between the neural
recording and the behaviour clock (the `video_gap` the description names) is in no file; the data
put it at 4 to 45 s in 25 of the 26 files where it can be estimated (294 s in the other), differing by
session. Every cluster is
labelled `unsorted` (5,879 against the paper's 998 cells), so the paper's cells cannot be selected.
The "ephys" files have no port entries, labels or poses; the BLA files have poses with every value
missing; no file holds alone sessions, phototagging pulses or the optogenetic experiments. The tone
schedule is the program's, the same in all files of a cohort, deliveries are 0.49 s after onset against the paper's 2 s,
`pair_in_dataset` says "not included" where the competitor's file is present, one session start
time and one subject description contradict the file names.